# 1C Structured Output and Tool Calling

**Purpose:**
Get answers from your endpoint that a program can use directly: JSON in a fixed shape, and calls to your own functions.

**What we're doing**

We ask for JSON the plain way and see it break, then have vLLM enforce a JSON schema. Then we give the model a Python function as a tool and run the call it asks for.

**Why we're doing it**

* Applications parse model output. Free-form text that only *looks* like JSON fails at the worst moment
* vLLM can constrain generation so the output always matches a schema, even on a 2B model
* Tool calling depends on two arguments you copied from the recipe in Section 1.3. This is where they pay off

**What to look for**

* Output that parses on every run, with exactly the fields you asked for
* A `tool_calls` field in the reply instead of text
* The difference between a reply that has the right *shape* and one that has the right *facts*

**Output**

A parsed, schema-checked JSON object, a small classifier, and a completed tool call.

> **Prerequisite:** Section 1 is complete: the `qwen-lab` instance is **Active** and `.env` exists at the project root. The instance was launched with the recipe's `--enable-auto-tool-choice --tool-call-parser qwen3_coder --reasoning-parser qwen3` arguments.


## 1C.1 Load Values from `.env`

Run the cell below. It reads the `.env` file you created in Section 1.6, so the endpoint, model name, and token are the same ones you used there.

In [ ]:
import sys
sys.path.insert(0, "..")
from config import KLIQUE_BASE_URL as base_url, KLIQUE_MODEL as model, KLIQUE_GATEWAY_TOKEN as token
import requests

url_models = f"{base_url}/models"
url_chat = f"{base_url}/chat/completions"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json",
}

print(f"Endpoint: {base_url}")
print(f"Model:    {model}")
print(f"Token:    {token[:8]}...")

import json


def chat(messages, max_tokens=512, **extra):
    """Send one chat completion and return the parsed response, or None on an error."""
    data = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "chat_template_kwargs": {"enable_thinking": False},
        **extra,
    }
    response = requests.post(url_chat, headers=headers, json=data, timeout=120)
    if not response.ok:
        print(f"❌ HTTP {response.status_code}: {response.text[:300]}")
        return None
    return response.json()

## 1C.2 Ask for JSON the Plain Way

First, just ask for JSON in the prompt. The model usually tries, but nothing makes it comply: it may wrap the JSON in a Markdown code fence, add a sentence before it, or rename a field. Run the cell a few times and watch whether the result parses.

In [ ]:
prompt = (
    "Describe the NVIDIA H100 GPU as JSON with exactly these keys: "
    "name (string), vendor (string), memory_gb (integer), good_for (list of up to 3 strings)."
)
reply = chat([{"role": "user", "content": prompt}])
if reply:
    text = reply["choices"][0]["message"]["content"]
    print(text)
    print()
    try:
        json.loads(text)
        print("✅ Parsed as JSON this time.")
    except json.JSONDecodeError as error:
        print(f"❌ Not valid JSON: {error}")

Even when it parses, you have no guarantee that the next reply will, or that the keys and types are the ones you asked for.

## 1C.3 Enforce a JSON Schema

vLLM accepts the OpenAI-style `response_format` option with a **JSON schema**. vLLM then restricts each generated token to ones that keep the output valid against the schema, so the reply is always parseable JSON with the fields you defined.

The schema below says what the object must contain. `additionalProperties: False` forbids extra keys.

In [ ]:
gpu_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "vendor": {"type": "string"},
        "memory_gb": {"type": "integer"},
        "good_for": {"type": "array", "items": {"type": "string"}, "maxItems": 3},
    },
    "required": ["name", "vendor", "memory_gb", "good_for"],
    "additionalProperties": False,
}

reply = chat(
    [{"role": "user", "content": "Describe the NVIDIA H100 GPU."}],
    response_format={"type": "json_schema", "json_schema": {"name": "gpu", "schema": gpu_schema}},
)
if reply:
    choice = reply["choices"][0]
    gpu = json.loads(choice["message"]["content"])
    print(json.dumps(gpu, indent=2))
    print(f"\nFinish reason: {choice['finish_reason']}")

The prompt no longer describes the format at all; the schema does that. The output parses directly with `json.loads`.

A program should still check the result before trusting it. The function below checks the keys and types without any extra library. If `finish_reason` is `length`, the reply was cut off by `max_tokens` before the JSON closed; raise `max_tokens` rather than trying to repair the text.

In [ ]:
def check_gpu(obj):
    """Return a list of problems with a GPU object; an empty list means it matches the schema."""
    problems = []
    if set(obj) != set(gpu_schema["required"]):
        problems.append(f"keys are {sorted(obj)}")
    if not isinstance(obj.get("name"), str) or not isinstance(obj.get("vendor"), str):
        problems.append("name and vendor must be strings")
    if not isinstance(obj.get("memory_gb"), int):
        problems.append("memory_gb must be an integer")
    good_for = obj.get("good_for")
    if not isinstance(good_for, list) or len(good_for) > 3 or not all(isinstance(x, str) for x in good_for):
        problems.append("good_for must be a list of up to 3 strings")
    return problems


if reply:
    problems = check_gpu(gpu)
    print("✅ Matches the schema." if not problems else f"❌ {problems}")

> **Note:** The schema guarantees the *shape*, not the *facts*. A 2B model can return a well-formed object with a wrong `memory_gb`. When a value matters, get it from a trusted source, which is what tool calling in 1C.5 is for.

## 1C.4 Classify with an Enum

A schema with an `enum` turns the model into a classifier whose answer is always one of your labels. The cell below sorts three support tickets.

In [ ]:
ticket_schema = {
    "type": "object",
    "properties": {
        "category": {"type": "string", "enum": ["billing", "bug", "feature_request", "other"]},
        "urgent": {"type": "boolean"},
    },
    "required": ["category", "urgent"],
    "additionalProperties": False,
}

tickets = [
    "I was charged twice for my subscription this month. Please refund one charge.",
    "The export button crashes the app every time since this morning's update. Our demo is in an hour!",
    "It would be great if the dashboard had a dark mode.",
]

for ticket in tickets:
    reply = chat(
        [
            {"role": "system", "content": "Classify the support ticket."},
            {"role": "user", "content": ticket},
        ],
        max_tokens=64,
        response_format={"type": "json_schema", "json_schema": {"name": "ticket", "schema": ticket_schema}},
    )
    if reply:
        label = json.loads(reply["choices"][0]["message"]["content"])
        print(f"{label['category']:<16} urgent={str(label['urgent']):<5}  {ticket[:60]}")

> **Checkpoint:** every line shows one of the four categories. The labels may not always be the ones you would pick, but none falls outside the enum.

## 1C.5 Let the Model Call a Function

**Tool calling** lets the model ask your code to do something, such as look up a value, instead of guessing. You describe each function, its name, purpose, and parameters, in the request's `tools` list. When the model decides it needs one, the reply's message has a `tool_calls` field instead of text. Your code runs the function and sends the result back, and the model writes the final answer.

Two recipe arguments from Section 1.3 make this work:

| vLLM argument | What it does |
|---|---|
| `--enable-auto-tool-choice` | Lets the model decide for itself when to call a tool (`"tool_choice": "auto"`) |
| `--tool-call-parser qwen3_coder` | Turns Qwen's tool-call text into the structured `tool_calls` field |

The tool below looks up GPU memory from a small table that you control, so the answer's numbers come from your data, not from the model's memory.

In [ ]:
GPU_MEMORY_GB = {"H100": 80, "H200": 141, "A100": 80, "L4": 24}


def lookup_gpu_memory(gpu):
    """The tool: return a GPU's memory in GB from our own table."""
    key = gpu.upper().replace("NVIDIA", "").strip()
    if key in GPU_MEMORY_GB:
        return {"gpu": key, "memory_gb": GPU_MEMORY_GB[key]}
    return {"error": f"Unknown GPU {gpu!r}. Known GPUs: {', '.join(GPU_MEMORY_GB)}"}


tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_gpu_memory",
            "description": "Look up how much memory an NVIDIA data center GPU has, in GB.",
            "parameters": {
                "type": "object",
                "properties": {"gpu": {"type": "string", "description": "GPU model, for example H100 or L4"}},
                "required": ["gpu"],
            },
        },
    }
]
TOOLS = {"lookup_gpu_memory": lookup_gpu_memory}

The loop below sends the question, runs every tool call the model asks for, returns the results as `tool` messages, and repeats until the model answers in text. `MAX_ROUNDS` bounds it, so a model that keeps calling tools cannot loop forever.

In [ ]:
MAX_ROUNDS = 3
messages = [{"role": "user", "content": "Which has more memory, an H100 or an L4, and by how many GB?"}]

for round_number in range(1, MAX_ROUNDS + 1):
    reply = chat(messages, tools=tools, tool_choice="auto")
    if not reply:
        break
    message = reply["choices"][0]["message"]
    calls = message.get("tool_calls") or []
    if not calls:
        print(f"✅ Answer after {round_number - 1} tool round(s):\n{message['content']}")
        break
    messages.append({"role": "assistant", "content": message.get("content"), "tool_calls": calls})
    for call in calls:
        name = call["function"]["name"]
        try:
            arguments = json.loads(call["function"]["arguments"])
            result = TOOLS[name](**arguments)
        except (KeyError, TypeError, json.JSONDecodeError) as error:
            result = {"error": f"Could not run {name}: {error}"}
        print(f"Round {round_number}: {name}({call['function']['arguments']}) -> {result}")
        messages.append({"role": "tool", "tool_call_id": call["id"], "content": json.dumps(result)})
else:
    print(f"Stopped after {MAX_ROUNDS} rounds without a final answer.")

You should see one or two `lookup_gpu_memory` calls, often both in the same round, followed by an answer that uses the numbers from the table: 80 GB and 24 GB, a difference of 56 GB.

> **Checkpoint:** at least one `Round` line and a final `✅ Answer`. To see the model skip the tool, change the question to one it can answer without a lookup, such as *What does GPU stand for?*

This is the core of an agent. Section 4 builds a full one, with several tools and step limits, and runs it through the LLM Gateway so that every step is metered and governed.

## 1C.6 See the Model's Reasoning

Every request in this notebook turned thinking off. Turn it on to see what `--reasoning-parser qwen3` does: the model's thinking arrives in its own field, separate from the answer, so `content` stays clean for your program. Thinking produces many more tokens, so this request allows more of them. Older vLLM versions name the field `reasoning_content` instead of `reasoning`; the cell checks both.

In [ ]:
data = {
    "model": model,
    "messages": [{"role": "user", "content": "Is 391 a prime number? Answer yes or no, then give the reason in one sentence."}],
    "max_tokens": 2048,
    "chat_template_kwargs": {"enable_thinking": True},
}
response = requests.post(url_chat, headers=headers, json=data, timeout=120)
if response.ok:
    body = response.json()
    message = body["choices"][0]["message"]
    reasoning = message.get("reasoning") or message.get("reasoning_content") or ""
    print(f"Reasoning ({len(reasoning)} characters), first 500:\n{reasoning[:500]}\n")
    print(f"Answer:\n{message['content']}\n")
    print(f"Finish reason: {body['choices'][0]['finish_reason']}, completion tokens: {body['usage']['completion_tokens']}")
else:
    print(f"❌ HTTP {response.status_code}: {response.text[:300]}")

If `content` is empty and `finish_reason` is `length`, the model spent the whole budget thinking. That is why Section 1 turned thinking off for a quick test. (391 is 17 × 23, so the answer is no.)

**If you are continuing to Section 1D or Section 2, leave the instance running.** Otherwise, stop it as described in Section 1.9.


## Troubleshooting

| Symptom | Likely cause |
|---|---|
| `400` mentioning `response_format` or the schema | The schema uses a feature your vLLM version does not support. Start from the schemas in this notebook and change one thing at a time. |
| `json.loads` fails with a schema set | The reply was cut off: check that `finish_reason` is `length` and raise `max_tokens`. |
| `400` saying `"auto" tool choice requires --enable-auto-tool-choice and --tool-call-parser` | The instance was launched without those recipe arguments. Relaunch it with the full arguments from Section 1.3. |
| The model answers in text without calling the tool | The question did not need a lookup, or the tool description does not match it. Make the description say when to use the tool. |
| Tool-call arguments do not parse | Small models sometimes produce malformed arguments. The loop returns an error result to the model, which usually corrects itself in the next round. |
| Connection error or timeout | The instance stopped, often from the idle time limit set in 1.3. Check its status and relaunch it. |

## Next: Tune the Deployment

So far you have used the recipe's arguments unchanged. Next, change them on purpose and see what each change does.

**Next:** In `01d_Tune_and_Relaunch_a_Deployment.ipynb` we limit the context length and the number of concurrent sequences, relaunch the instance, and measure the effect.